# Import Libraries

In [4]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os


## CONFIGURATION

In [5]:
DATA_PATH = "data/eye_dataset_v2.csv"

MAX_SAMPLES_PER_CLASS = 500

# هر چند فریم یک نمونه ذخیره شود
FRAME_INTERVAL = 3


### CREATE DATA DIRECTORY

In [6]:
os.makedirs("data", exist_ok=True)

### MEDIAPIPE

In [7]:

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


# EYE LANDMARKS

In [8]:
LEFT_EYE = [
    362, 385, 387,
    263, 373, 380
]

RIGHT_EYE = [
    33, 160, 158,
    133, 153, 144
]



#### FEATURE EXTRACTION

In [9]:
def get_eye_features(landmarks, eye_indices):

    points = []

    for index in eye_indices:

        x = landmarks[index].x
        y = landmarks[index].y

        points.append(
            np.array([x, y])
        )

    p1, p2, p3, p4, p5, p6 = points

    vertical_1 = np.linalg.norm(
        p2 - p6
    )

    vertical_2 = np.linalg.norm(
        p3 - p5
    )

    horizontal = np.linalg.norm(
        p1 - p4
    )

    if horizontal == 0:

        return None

    ear = (
        vertical_1 + vertical_2
    ) / (
        2.0 * horizontal
    )

    return {

        "ear": ear,

        "width": horizontal,

        "height_1": vertical_1,

        "height_2": vertical_2
    }



# CAMERA

In [10]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():

    raise RuntimeError(
        "Cannot access webcam."
    )



data = []

open_count = 0
closed_count = 0

collecting = None

frame_counter = 0


print("=" * 70)
print("AUTOMATIC EYE DATASET COLLECTION")
print("=" * 70)

print()
print("CLICK ON THE CAMERA WINDOW FIRST")
print()
print("O -> Collect OPEN automatically")
print("C -> Collect CLOSED automatically")
print("Q -> Quit")
print()
print("Only press O or C ONCE.")
print("=" * 70)


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:

        print("Cannot read webcam frame.")

        break


    frame = cv2.flip(
        frame,
        1
    )


    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    results = face_mesh.process(
        rgb
    )


    # ========================================================
    # FACE DETECTION
    # ========================================================

    if results.multi_face_landmarks:

        face = results.multi_face_landmarks[0]

        landmarks = face.landmark


        left = get_eye_features(
            landmarks,
            LEFT_EYE
        )

        right = get_eye_features(
            landmarks,
            RIGHT_EYE
        )


        if left is not None and right is not None:

            left_ear = left["ear"]

            right_ear = right["ear"]

            ear = (
                left_ear +
                right_ear
            ) / 2.0


            # =================================================
            # FEATURES
            # =================================================

            features = {

                "left_ear":
                    left_ear,

                "right_ear":
                    right_ear,

                "ear":
                    ear,

                "left_eye_width":
                    left["width"],

                "left_eye_height_1":
                    left["height_1"],

                "left_eye_height_2":
                    left["height_2"],

                "right_eye_width":
                    right["width"],

                "right_eye_height_1":
                    right["height_1"],

                "right_eye_height_2":
                    right["height_2"]
            }


            # =================================================
            # AUTOMATIC COLLECTION
            # =================================================

            if collecting is not None:

                frame_counter += 1


                if frame_counter >= FRAME_INTERVAL:

                    if collecting == "OPEN":

                        if open_count < MAX_SAMPLES_PER_CLASS:

                            row = features.copy()

                            row["label"] = "OPEN"

                            data.append(row)

                            open_count += 1

                            print(
                                f"OPEN: {open_count}/500"
                            )


                    elif collecting == "CLOSED":

                        if closed_count < MAX_SAMPLES_PER_CLASS:

                            row = features.copy()

                            row["label"] = "CLOSED"

                            data.append(row)

                            closed_count += 1

                            print(
                                f"CLOSED: {closed_count}/500"
                            )


                    frame_counter = 0


                # ---------------------------------------------
                # OPEN COMPLETE
                # ---------------------------------------------

                if (
                    collecting == "OPEN"
                    and open_count >= MAX_SAMPLES_PER_CLASS
                ):

                    collecting = None

                    print()
                    print(
                        "OPEN COLLECTION COMPLETE"
                    )

                    print(
                        "Now close your eyes and press C once."
                    )


                # ---------------------------------------------
                # CLOSED COMPLETE
                # ---------------------------------------------

                if (
                    collecting == "CLOSED"
                    and closed_count >= MAX_SAMPLES_PER_CLASS
                ):

                    collecting = None

                    print()
                    print(
                        "CLOSED COLLECTION COMPLETE"
                    )


            # =================================================
            # DISPLAY
            # =================================================

            cv2.putText(
                frame,
                f"EAR: {ear:.3f}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"OPEN: {open_count}/500",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"CLOSED: {closed_count}/500",
                (20, 115),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 0),
                2
            )


            if collecting is not None:

                cv2.putText(
                    frame,
                    f"COLLECTING: {collecting}",
                    (20, 155),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2
                )


    # ========================================================
    # SHOW FRAME
    # ========================================================

    cv2.imshow(
        "Automatic Eye Dataset Collection",
        frame
    )


    # ========================================================
    # KEYBOARD INPUT
    # ========================================================

    key = cv2.waitKey(1) & 0xFF


    # --------------------------------------------------------
    # Q
    # --------------------------------------------------------

    if key == ord("q"):

        print()
        print("Q pressed -> stopping...")

        break


    # --------------------------------------------------------
    # O
    # --------------------------------------------------------

    elif key == ord("o"):

        if open_count < MAX_SAMPLES_PER_CLASS:

            collecting = "OPEN"

            frame_counter = 0

            print()
            print(
                ">>> START OPEN COLLECTION <<<"
            )


    # --------------------------------------------------------
    # C
    # --------------------------------------------------------

    elif key == ord("c"):

        if closed_count < MAX_SAMPLES_PER_CLASS:

            collecting = "CLOSED"

            frame_counter = 0

            print()
            print(
                ">>> START CLOSED COLLECTION <<<"
            )


# ============================================================
# CLEANUP
# ============================================================

cap.release()

cv2.destroyAllWindows()

face_mesh.close()


# ============================================================
# SAVE
# ============================================================

if len(data) > 0:

    df = pd.DataFrame(data)


    columns = [

        "left_ear",
        "right_ear",
        "ear",

        "left_eye_width",

        "left_eye_height_1",
        "left_eye_height_2",

        "right_eye_width",

        "right_eye_height_1",
        "right_eye_height_2",

        "label"
    ]


    df = df[columns]


    df.to_csv(
        DATA_PATH,
        index=False
    )


    print()
    print("=" * 70)
    print("DATASET SAVED")
    print("=" * 70)

    print(
        f"Total samples : {len(df)}"
    )

    print(
        f"OPEN          : {open_count}"
    )

    print(
        f"CLOSED        : {closed_count}"
    )

    print(
        f"File          : {DATA_PATH}"
    )

    print("=" * 70)


else:

    print("No data collected.")

AUTOMATIC EYE DATASET COLLECTION

CLICK ON THE CAMERA WINDOW FIRST

O -> Collect OPEN automatically
C -> Collect CLOSED automatically
Q -> Quit

Only press O or C ONCE.

Q pressed -> stopping...
No data collected.
